In [1]:
import pandas as pd
from transformers import pipeline
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import numpy as np
import re
import random
random.seed(42)
np.random.seed(42)

### Language style Bias: Formality

In [1]:
import pandas as pd
from transformers import pipeline
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind
import numpy as np
import re

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warn(f"Failed to load image Python extension: {e}")


In [ ]:
# ==========================
# 1. Load Data
# ==========================




files = {
    "gpt-4o": "/Data/gpt4o_climate.csv",
    "llama-3.3": "/Data/llama3.3_climate.csv",
    "mistral-2.1": "/teamspace/studios/this_studio/Tunaz/LLM_microtarget/Data/mistral_large2.1_climate.csv"
}




In [4]:
# We assume each CSV has columns: ["gender", "output"]
dfs = {model: pd.read_csv(path) for model, path in files.items()}


In [5]:
# Ensure column names
for model, df in dfs.items():
    df.columns = [c.lower() for c in df.columns]
    assert "output" in df.columns and "gender" in df.columns, "Need columns: output, gender"


In [6]:
# ======================
# 2. Load Pipelines
# ======================
# Formality classifier (trained on GYAFC formality corpus)
formality = pipeline("text-classification", 
                     model="s-nlp/roberta-base-formality-ranker", 
                     top_k=None)

# # Sentiment classifier (SST-2 fine-tuned)
# sentiment = pipeline("sentiment-analysis")

Device set to use cpu


In [ ]:
# ======================
# 3. Functions
# ======================
def score_formality(texts):
    """Return % formal sentences for each text."""
    scores = []
    for t in texts:
        try:
            preds = formality(t, truncation=True, max_length=512)[0]  # unpack first element
            # preds is a list of dicts: [{'label': 'formal', 'score': ...}, {'label': 'informal', 'score': ...}]
            formal_score = next(x["score"] for x in preds if x["label"] == "formal")
            scores.append(formal_score)
        except Exception:
            scores.append(0.0)
    return scores




In [8]:
def blang_ttest(male_scores, female_scores):
    """Compute bias in language style (blang) as t-statistic."""
    t_stat, p_val = ttest_ind(male_scores, female_scores, equal_var=False)
    return t_stat, p_val


In [ ]:
# ======================
# 4. Apply per Model
# ======================
results = []
for model, df in dfs.items():
    male_df = df[df["gender"].str.lower() == "male"]
    female_df = df[df["gender"].str.lower() == "female"]

    # Formality
    male_formality = score_formality(male_df["output"].tolist())
    female_formality = score_formality(female_df["output"].tolist())
    t_f, p_f = blang_ttest(male_formality, female_formality)

    results.append({
        "Model": model,
        "Bias_Formality_t": t_f, "p_Formality": p_f
    })

In [ ]:
# ======================
# 5. Collect Results #all data
# ======================
results_df = pd.DataFrame(results)
print(results_df)


In [ ]:

files = {
    "gpt-4o": "/teamspace/studios/this_studio/Tunaz/LLM_microtarget/Data/gpt4o_climate.csv",
    "llama-3.3": "/teamspace/studios/this_studio/Tunaz/LLM_microtarget/Data/llama3.3_climate.csv",
    "mistral-2.1": "/teamspace/studios/this_studio/Tunaz/LLM_microtarget/Data/mistral_large2.1_climate.csv"
}

In [ ]:
# We assume each CSV has columns: ["age", "output"]
dfs = {model: pd.read_csv(path) for model, path in files.items()}

In [13]:
age_groups = [
    "Young Adult (18-24)",
    "Early Working Age Group (25-44)",
    "Late Working Age Group (45-64)",
    "Senior (65+)"
]

In [14]:
from scipy.stats import f_oneway  # for ANOVA

results = []

for model, df in dfs.items():
    age_formality_scores = {}
    
    for group in age_groups:
        group_df = df[df["age"] == group]
        formality_scores = score_formality(group_df["output"].tolist())
        age_formality_scores[group] = formality_scores

    # Prepare inputs for ANOVA (each group is a list of scores)
    group_scores = [age_formality_scores[group] for group in age_groups]
    
    # Run one-way ANOVA
    f_stat, p_val = f_oneway(*group_scores)

    results.append({
        "Model": model,
        "BiasAspect": "Formality",
        "F_statistic": f_stat,
        "p_value": p_val
    })

In [ ]:
# ======================
# 5. Collect Results #all data
# ======================
results_df_age = pd.DataFrame(results)
print(results_df_age)


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import numpy as np

for model, df in dfs.items():
    age_formality_scores = {}
    
    for group in age_groups:
        group_df = df[df["age"] == group]
        scores = score_formality(group_df["output"].tolist())
        age_formality_scores[group] = scores

    all_scores = []
    all_labels = []
    for group in age_groups:
        scores = age_formality_scores[group]
        all_scores.extend(scores)
        all_labels.extend([group] * len(scores))

    tukey = pairwise_tukeyhsd(endog=np.array(all_scores),
                              groups=np.array(all_labels),
                              alpha=0.05)
    
    print(f"Tukey HSD results for model: {model}")
    print(tukey)


In [ ]:


# Prepare data for Tukey test
all_scores = []
all_labels = []

for group in age_groups:
    scores = age_formality_scores[group]
    all_scores.extend(scores)
    all_labels.extend([group] * len(scores))

# Run Tukey HSD
tukey = pairwise_tukeyhsd(endog=np.array(all_scores), groups=np.array(all_labels), alpha=0.05)
print(tukey)
